In [0]:
df_mara = spark.table("sap_sd_project.bronze.mara_raw")
display(df_mara)

MANDT,MATNR,MTART,MATKL,MEINS,BRGEW,NTGEW,GEWEI,ERSDA
100,MAT00001,FERT,ELEC,Each,8.722,7.013,KG,2022-08-11
100,MAT00002,fert,OFFC,EA,7.454,6.988,KG,2024-09-05
100,MAT00003,ROH,PACK,EA,24.594,22.061,KG,2021-09-03
100,MAT00004,HAWA,MRO,EA,19.013,16.483,KG,2022-05-29
100,MAT00005,FERT,ELEC,EA,18.634,15.904,KG,2020-01-06
100,MAT00006,FERT,OFFC,EA,29.509,27.085,KG,2021-05-29
100,MAT00007,ROH,null,EA,-12.104,10.804,KG,2022-06-22
100,MAT00008,HAWA,MRO,EA,25.073,20.9,KG,2022-06-15
100,MAT00009,FERT,ELEC,EA,33.475,25.523,KG,2020-05-26
100,MAT00010,FERT,OFFC,EA,3.479,2.942,KG,2024-03-03


In [0]:
df_mara.printSchema()

root
 |-- MANDT: long (nullable = true)
 |-- MATNR: string (nullable = true)
 |-- MTART: string (nullable = true)
 |-- MATKL: string (nullable = true)
 |-- MEINS: string (nullable = true)
 |-- BRGEW: double (nullable = true)
 |-- NTGEW: double (nullable = true)
 |-- GEWEI: string (nullable = true)
 |-- ERSDA: date (nullable = true)



In [0]:
from pyspark.sql.functions import trim, col

display(
    df_mara
    .withColumn("MATNR_CLEAN", trim(col("MATNR")))
    .groupBy("MANDT", "MATNR_CLEAN")
    .count()
    .filter(col("count") > 1)
)

MANDT,MATNR_CLEAN,count
100,MAT00026,2
100,MAT00053,2
100,MAT00065,2
100,MAT00071,2


In [0]:
df_mara_clean = (
    df_mara
    .withColumn("MATNR",trim(col("MATNR")))
)

In [0]:
df_mara_clean = (
    df_mara_clean
    .dropDuplicates(["MANDT","MATNR"])
)

In [0]:
display(
    df_mara_clean
    .groupBy("MANDT","MATNR")
    .count()
    .filter(col("count") > 1)
)

MANDT,MATNR,count


In [0]:
from pyspark.sql.functions import upper, col
df_mara_clean = (
    df_mara_clean
    .withColumn("MTART",upper(col("MTART")))
)

In [0]:
display(
    df_mara_clean
    .select("MTART")
    .distinct()
    .orderBy("MTART")
)

MTART
FERT
HAWA
ROH


In [0]:
from pyspark.sql.functions import col,when

df_mara_clean = (
    df_mara_clean
    .withColumn(
        "MATKL",
        when(col("MATKL").isNull(), "UNKNOWN")
        .otherwise(col("MATKL"))
    )
)

In [0]:
display(
    df_mara_clean
    .select("MATKL")
    .distinct()
    .orderBy("MATKL")
)

MATKL
ELEC
MRO
OFFC
PACK
UNKNOWN


In [0]:
from pyspark.sql.functions import col, trim, upper, when

df_mara_clean = (
    df_mara_clean
    .withColumn("MEINS", upper(trim(col("MEINS"))))
    .withColumn(
        "MEINS",
        when(col("MEINS").isin("EA", "EACH", "PC"), "EA")
        .otherwise(col("MEINS"))
    )
)

In [0]:
display(
    df_mara_clean
    .select("MEINS")
    .distinct()
)

MEINS
EA


In [0]:
from pyspark.sql.functions import col, when

df_mara_clean = (
    df_mara_clean
    .withColumn(
        "BRGEW_INVALID_FLAG",
        when(col("BRGEW") < 0, 1).otherwise(0)
    )
    .withColumn(
        "BRGEW",
        when(col("BRGEW") < 0, None)
        .otherwise(col("BRGEW"))
    )
)

In [0]:
display(
    df_mara_clean
    .filter(col("BRGEW_INVALID_FLAG") == 1)
    .select("MATNR", "BRGEW", "BRGEW_INVALID_FLAG")
)

MATNR,BRGEW,BRGEW_INVALID_FLAG
MAT00007,null,1
MAT00018,null,1
MAT00053,null,1


In [0]:
from pyspark.sql.functions import col, when

df_mara_clean = (
    df_mara_clean
    .withColumn(
        "NTGEW_MISSING_FLAG",
        when(col("NTGEW").isNull(), 1).otherwise(0)
    )
)

In [0]:
display(
    df_mara_clean
    .filter(col("NTGEW_MISSING_FLAG") == 1)
    .select(
        "MATNR",
        "BRGEW",
        "NTGEW",
        "NTGEW_MISSING_FLAG"
    )
)

MATNR,BRGEW,NTGEW,NTGEW_MISSING_FLAG
MAT00023,25.899,null,1
MAT00052,5.449,null,1
MAT00053,null,null,1


In [0]:
display(
    df_mara_clean
    .filter(
        col("NTGEW").isNotNull() &
        col("BRGEW").isNotNull() &
        (col("NTGEW") > col("BRGEW"))
    )
    .select(
        "MATNR",
        "BRGEW",
        "NTGEW"
    )
)

MATNR,BRGEW,NTGEW


In [0]:
print("Rows:", df_mara_clean.count())

display(
    df_mara_clean.select(
        "MATNR",
        "MTART",
        "MATKL",
        "MEINS",
        "BRGEW",
        "NTGEW",
        "BRGEW_INVALID_FLAG",
        "NTGEW_MISSING_FLAG"
    )
)

Rows: 80


MATNR,MTART,MATKL,MEINS,BRGEW,NTGEW,BRGEW_INVALID_FLAG,NTGEW_MISSING_FLAG
MAT00001,FERT,ELEC,EA,8.722,7.013,0,0
MAT00002,FERT,OFFC,EA,7.454,6.988,0,0
MAT00003,ROH,PACK,EA,24.594,22.061,0,0
MAT00004,HAWA,MRO,EA,19.013,16.483,0,0
MAT00005,FERT,ELEC,EA,18.634,15.904,0,0
MAT00006,FERT,OFFC,EA,29.509,27.085,0,0
MAT00007,ROH,UNKNOWN,EA,null,10.804,1,0
MAT00008,HAWA,MRO,EA,25.073,20.9,0,0
MAT00009,FERT,ELEC,EA,33.475,25.523,0,0
MAT00010,FERT,OFFC,EA,3.479,2.942,0,0


In [0]:
df_mara_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sap_sd_project.silver.mara_clean")

In [0]:
display(
    spark.table("sap_sd_project.silver.mara_clean")
)

MANDT,MATNR,MTART,MATKL,MEINS,BRGEW,NTGEW,GEWEI,ERSDA,BRGEW_INVALID_FLAG,NTGEW_MISSING_FLAG
100,MAT00021,FERT,ELEC,EA,19.142,17.231,KG,2024-02-13,0,0
100,MAT00023,ROH,PACK,EA,25.899,null,KG,2023-12-19,0,1
100,MAT00024,HAWA,MRO,EA,0.241,0.213,KG,2023-06-12,0,0
100,MAT00039,ROH,PACK,EA,8.045,7.791,KG,2021-07-01,0,0
100,MAT00049,FERT,ELEC,EA,31.105,25.375,KG,2022-06-21,0,0
100,MAT00051,ROH,PACK,EA,22.109,16.787,KG,2020-01-05,0,0
100,MAT00067,ROH,PACK,EA,30.419,24.748,KG,2019-12-15,0,0
100,MAT00075,ROH,PACK,EA,34.131,31.369,KG,2021-04-10,0,0
100,MAT00005,FERT,ELEC,EA,18.634,15.904,KG,2020-01-06,0,0
100,MAT00013,FERT,ELEC,EA,24.892,22.824,KG,2021-07-04,0,0
